In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("amohankumar/bone-break-classifier-dataset")

print("Path to dataset files:", path)

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
amohankumar_bone_break_classifier_dataset_path = kagglehub.dataset_download('amohankumar/bone-break-classifier-dataset')

print('Data source import complete.')


In [ ]:
import os
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models
from torchvision.utils import make_grid
!pip install pytorch-lightning
import pytorch_lightning as pl
from pytorch_lightning import LightningModule, Trainer
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
from PIL import Image
import zipfile  # Agrega esta línea
%matplotlib inline

In [ ]:
# Definir la transformación de datos
transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
# Crear el conjunto de datos
dataset = datasets.ImageFolder(root="/kaggle/input/bone-break-classifier-dataset", transform=transform)

# Clases del conjunto de datos
class_names = dataset.classes


print(class_names)
print(len(class_names))

In [ ]:
# Crear el directorio "datasets" si no existe
!mkdir datasets


In [ ]:
# Funciones para comprimir y extraer archivos zip
def create_zip(source_folder, destination_zip):
    # Remove existing zip file if it exists, to avoid potential corruption/lock issues
    if os.path.exists(destination_zip):
        os.remove(destination_zip)

    # Check if source_folder exists and is a directory
    if not os.path.isdir(source_folder):
        print(f"Warning: Source folder does not exist or is not a directory: {source_folder}. Skipping zipping.")
        return

    with zipfile.ZipFile(destination_zip, 'w', zipfile.ZIP_DEFLATED) as zip_ref:
        files_found = False
        for root, dirs, files in os.walk(source_folder):
            for file in files:
                files_found = True
                file_path = os.path.join(root, file)
                zip_ref.write(file_path, arcname=os.path.relpath(file_path, source_folder))

        if not files_found:
            print(f"Warning: No files found in source folder: {source_folder}. Creating an empty zip file.")

def extract_zip(zip_file, destination_folder):
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(destination_folder)

In [ ]:
# Crear y extraer archivos zip para cada clase

# Ensure the output directory for zips exists
zip_output_dir = '/kaggle/working/zips'
os.makedirs(zip_output_dir, exist_ok=True)

for name in class_names:
    name2 = name.replace(' ', '_')
    # Corrected source_folder path
    source_folder = os.path.join(amohankumar_bone_break_classifier_dataset_path, name)
    destination_zip = os.path.join(zip_output_dir, name2 + '_folder.zip')
    destination_folder = '/kaggle/working/datasets/' + name2

    create_zip(source_folder, destination_zip)
    extract_zip(destination_zip, destination_folder)


In [ ]:
!ls datasets


In [ ]:
# Clase para el módulo de datos
class DataModule(pl.LightningDataModule):
    def __init__(self, transform=transform, batch_size=32):
        super().__init__()
        self.root_dir = "/kaggle/working/datasets"
        self.transform = transform
        self.batch_size = batch_size

    def setup(self, stage=None):
        dataset = datasets.ImageFolder(root=self.root_dir, transform=self.transform)
        n_data = len(dataset)
        n_train = int(0.6 * n_data)
        n_valid = int(0.2 * n_data)
        n_test = n_data - n_train - n_valid

        train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(dataset, [n_train, n_valid, n_test])

        self.train_dataset = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        self.val_dataset = DataLoader(val_dataset, batch_size=self.batch_size)
        self.test_dataset = DataLoader(test_dataset, batch_size=self.batch_size)

    def train_dataloader(self):
        return self.train_dataset

    def val_dataloader(self):
        return self.val_dataset

    def test_dataloader(self):
        return self.test_dataset

In [ ]:

# Clase del modelo MobileNetV2
class MobileNetV2Model(LightningModule):
    def __init__(self, num_classes=len(class_names)):
        super(MobileNetV2Model, self).__init__()

        # Cargar el modelo MobileNetV2 preentrenado
        self.mobilenet = models.mobilenet_v2()

        # Ajustar la capa clasificadora para el número de clases en tu conjunto de datos
        in_features = self.mobilenet.classifier[1].in_features
        self.mobilenet.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, X):
        return self.mobilenet(X)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=0.001)
        return optimizer

    def training_step(self, train_batch, batch_idx):
        X, y = train_batch
        y_hat = self(X)
        loss = F.cross_entropy(y_hat, y)
        pred = y_hat.argmax(dim=1, keepdim=True)
        acc = pred.eq(y.view_as(pred)).sum().item() / y.shape[0]
        self.log("train_loss", loss)
        self.log("train_acc", acc)
        return loss

    def validation_step(self, val_batch, batch_idx):
        X, y = val_batch
        y_hat = self(X)
        loss = F.cross_entropy(y_hat, y)
        pred = y_hat.argmax(dim=1, keepdim=True)
        acc = pred.eq(y.view_as(pred)).sum().item() / y.shape[0]
        self.log("val_loss", loss)
        self.log("val_acc", acc)

    def test_step(self, test_batch, batch_idx):
        X, y = test_batch
        y_hat = self(X)
        loss = F.cross_entropy(y_hat, y)
        pred = y_hat.argmax(dim=1, keepdim=True)
        acc = pred.eq(y.view_as(pred)).sum().item() / y.shape[0]
        self.log("test_loss", loss)
        self.log("test_acc", acc)

In [ ]:
# Crear el módulo de datos y el modelo
datamodule = DataModule()
datamodule.setup()
model = MobileNetV2Model()


In [ ]:
import pytorch_lightning as pl

# Crear el módulo de datos y el modelo
datamodule = DataModule()
datamodule.setup()
model = MobileNetV2Model()

# Entrenar el modelo
trainer = pl.Trainer(max_epochs=30)
trainer.fit(model, datamodule)

In [ ]:
# Evaluar el modelo
datamodule.setup(stage='test')
test_loader = datamodule.test_dataloader()
trainer.test(dataloaders=test_loader)


In [ ]:
# Visualizar algunas imágenes
for images, labels in datamodule.train_dataloader():
    break


In [ ]:
# Mostrar una cuadrícula de imágenes
im = make_grid(images, nrow=8)
plt.figure(figsize=(12, 12))
plt.imshow(np.transpose(im.numpy(), (1, 2, 0)))


In [ ]:
# Inversa de la normalización para la visualización
inv_normalize = transforms.Normalize(mean=[-0.485 / 0.229, -0.456 / 0.224, -0.406 / 0.225],
                                     std=[1 / 0.229, 1 / 0.224, 1 / 0.225])
im = inv_normalize(im)

plt.figure(figsize=(12, 12))
plt.imshow(np.transpose(im.numpy(), (1, 2, 0)))


In [ ]:
# Evaluar el modelo
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for test_data in datamodule.test_dataloader():
        test_images, test_labels = test_data[0].to(device), test_data[1].to(device)
        pred = model(test_images).argmax(dim=1)
        for i in range(len(pred)):
            y_true.append(test_labels[i].item())
            y_pred.append(pred[i].item())


In [ ]:
# Imprimir el informe de clasificación
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

In [ ]:
# Guardar el modelo
torch.save(model.state_dict(), "/kaggle/working/model_weights.pth")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, EfficientNetB3, ResNet50, MobileNetV3Small
import matplotlib.pyplot as plt

IMG_SIZE = 224
BATCH_SIZE = 32

train_dir = "/kaggle/working/datasets"
val_dir = "/kaggle/working/datasets"

In [ ]:
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2 # Added for splitting data into train/val
)

val_gen = ImageDataGenerator(rescale=1./255, validation_split=0.2) # Added for splitting data into train/val

train_data = train_gen.flow_from_directory(
    train_dir, # Ensure 'train_dir' is '/kaggle/working/datasets' in the previous cell
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical', # Changed from 'binary' to 'categorical' for multi-class
    subset='training' # Specify subset for training data
)

val_data = val_gen.flow_from_directory(
    val_dir, # Ensure 'val_dir' is '/kaggle/working/datasets' in the previous cell
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical', # Changed from 'binary' to 'categorical' for multi-class
    subset='validation' # Specify subset for validation data
)

In [ ]:
def build_model(base_model):
    base_model.trainable = False  # freeze base

    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(len(class_names), activation='softmax')(x) # Changed to 12 neurons with softmax

    model = models.Model(inputs=base_model.input, outputs=output)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='categorical_crossentropy', # Changed to categorical_crossentropy
        metrics=['accuracy']
    )

    return model

In [ ]:
base = MobileNetV3Small(weights='imagenet', include_top=False, input_shape=(224,224,3))
model_mobile = build_model(base)

history_mobile = model_mobile.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

In [ ]:
base = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))
model_resnet = build_model(base)

history_resnet = model_resnet.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

In [ ]:
def alexnet():
    model = models.Sequential()

    model.add(layers.Conv2D(96, (11,11), strides=4, activation='relu', input_shape=(224,224,3)))
    model.add(layers.MaxPooling2D(3, strides=2))

    model.add(layers.Conv2D(256, (5,5), padding='same', activation='relu'))
    model.add(layers.MaxPooling2D(3, strides=2))

    model.add(layers.Conv2D(384, (3,3), padding='same', activation='relu'))
    model.add(layers.Conv2D(384, (3,3), padding='same', activation='relu'))
    model.add(layers.Conv2D(256, (3,3), padding='same', activation='relu'))
    model.add(layers.MaxPooling2D(3, strides=2))

    model.add(layers.Flatten())
    model.add(layers.Dense(4096, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(4096, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(len(class_names), activation='softmax')) # Changed to len(class_names) neurons with softmax activation

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy', # Changed to categorical_crossentropy
        metrics=['accuracy']
    )

    return model

model_alex = alexnet()

history_alex = model_alex.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

In [ ]:
def plot_history(histories, names):
    for history, name in zip(histories, names):
        plt.plot(history.history['val_accuracy'], label=name)

    plt.legend()
    plt.title("Model Comparison")
    plt.xlabel("Epochs")
    plt.ylabel("Validation Accuracy")
    plt.grid(True)
    plt.show()


plot_history(
    [history_resnet, history_mobile, history_alex],
    ["ResNet50", "MobileNetV3", "AlexNet"]
)

In [ ]:
# Unfreeze base model
base.trainable = True   # Unfreeze the entire base model

# Freeze most layers, train last 20
for layer in base.layers[:-20]:
    layer.trainable = False

model_resnet.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_resnet_ft = model_resnet.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

In [ ]:
model_mobile.layers[0].trainable = True # The build_model function sets the base model as the first layer

# Unfreeze the base model
base.trainable = True

# Freeze most layers, train last 20
for layer in base.layers[:-20]:
    layer.trainable = False

model_mobile.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_mobile_ft = model_mobile.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

In [ ]:
model_alex.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_alex_ft = model_alex.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

### Validation Metrics for Fine-Tuned Models

In [ ]:
def plot_history(histories, names):
    for history, name in zip(histories, names):
        plt.plot(history.history['val_accuracy'], label=name)

    plt.legend()
    plt.title("Model Comparison")
    plt.xlabel("Epochs")
    plt.ylabel("Validation Accuracy")
    plt.grid(True)
    plt.show()

print('Plotting Validation Accuracy for Fine-Tuned Models:')
plot_history(
    [history_resnet_ft, history_mobile_ft, history_alex_ft],
    ["ResNet50_FT", "MobileNetV3_FT", "AlexNet_FT"]
)

def plot_loss_history(histories, names):
    plt.figure(figsize=(10, 6))
    for history, name in zip(histories, names):
        plt.plot(history.history['val_loss'], label=name)

    plt.legend()
    plt.title("Model Comparison - Validation Loss")
    plt.xlabel("Epochs")
    plt.ylabel("Validation Loss")
    plt.grid(True)
    plt.show()

print('\nPlotting Validation Loss for Fine-Tuned Models:')
plot_loss_history(
    [history_resnet_ft, history_mobile_ft, history_alex_ft],
    ["ResNet50_FT", "MobileNetV3_FT", "AlexNet_FT"]
)

In [ ]:
#validation accuracy graph
plot_history(
    [history_resnet_ft, history_mobile_ft, history_alex_ft],
    ["ResNet50_FT", "MobileNetV3_FT", "AlexNet_FT"]
)

In [ ]:
# Evaluate fine-tuned ResNet50 model
print("Evaluating Fine-Tuned ResNet50:")
loss_resnet, accuracy_resnet = model_resnet.evaluate(val_data)
print(f"ResNet50 Fine-Tuned Validation Loss: {loss_resnet:.4f}")
print(f"ResNet50 Fine-Tuned Validation Accuracy: {accuracy_resnet:.4f}\n")

In [ ]:
# Evaluate fine-tuned MobileNetV3 model
print("Evaluating Fine-Tuned MobileNetV3:")
loss_mobile, accuracy_mobile = model_mobile.evaluate(val_data)
print(f"MobileNetV3 Fine-Tuned Validation Loss: {loss_mobile:.4f}")
print(f"MobileNetV3 Fine-Tuned Validation Accuracy: {accuracy_mobile:.4f}\n")

In [ ]:
# Evaluate fine-tuned AlexNet model
print("Evaluating Fine-Tuned AlexNet:")
loss_alex, accuracy_alex = model_alex.evaluate(val_data)
print(f"AlexNet Fine-Tuned Validation Loss: {loss_alex:.4f}")
print(f"AlexNet Fine-Tuned Validation Accuracy: {accuracy_alex:.4f}")